In [1]:
from dotenv import load_dotenv
from utils.agent_visualizer import (
    display_agent_response,
    print_activity,
    reset_activity_context,
    visualize_conversation,
)

from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient

load_dotenv()

# Notebook 전체에서 사용할 model 정의
# 우수한 계획 및 추론 능력을 위해 Opus 4.6 사용
MODEL = "claude-opus-4-6"
print(f"📋 Notebook configured to use: {MODEL}")

📋 Notebook configured to use: claude-opus-4-6


# 01 - Chief of Staff Agent

#### 소개

Notebook 00에서는 단순한 research agent를 구축했습니다. 이 Notebook에서는 종합적인 agent를 구축하기 위한 Claude Code SDK의 핵심 기능을 점진적으로 소개합니다. 각 기능에 대해 다음을 설명합니다:
- **무엇**: 기능이 무엇인지
- **왜**: 기능이 무엇을 할 수 있고 왜 사용하려 하는지
- **어떻게**: 사용법을 보여 주는 최소 구현

Claude Code에 익숙하다면 SDK가 동등한 기능을 제공하며 모든 Claude Code 기능을 programmatic한 headless 방식으로 활용할 수 있게 한다는 점을 알 수 있습니다.

#### 시나리오

이 Notebook 전체에서 막 $10M Series A를 유치한 50명 규모 startup을 위한 **AI Chief of Staff**를 구축합니다. CEO는 공격적인 성장과 재무 지속 가능성 사이에서 균형을 잡기 위해 data 기반 insight가 필요합니다.

최종 Chief of Staff agent는 다음을 수행합니다:
- 서로 다른 분야의 **전문 subagent 조정**
- 여러 출처의 **insight 종합**
- 실행 가능한 권고를 담은 **경영진용 요약 제공**

## 기본 기능

### 기능 0: [CLAUDE.md](https://www.anthropic.com/engineering/claude-code-best-practices)를 통한 Memory

**무엇**: `CLAUDE.md` 파일은 agent의 영속적인 memory와 지침 역할을 합니다. 프로젝트 디렉터리에 있으면 Claude Code는 agent를 초기화할 때 이 context를 자동으로 읽고 반영합니다.

**왜**: 상호작용마다 프로젝트 context, 팀 선호도, standard를 반복해서 제공하는 대신 `CLAUDE.md`에 한 번 정의할 수 있습니다. 일관된 동작을 보장하며 중복 설명을 피하여 token 사용을 줄입니다.

**어떻게**:
- 작업 디렉터리에 `CLAUDE.md` 파일을 둡니다. 예제에서는 `chief_of_staff_agent/CLAUDE.md`입니다
- ClaudeSDKClient의 `cwd` argument를 CLAUDE.md 파일의 디렉터리로 지정합니다
- 상세 data 파일보다 상위 수준 context를 우선하게 하려면 명시적인 prompt로 agent를 안내합니다

**중요한 동작 참고**: CLAUDE.md와 상세 data 파일(CSV 등)이 모두 있으면 agent는 정확한 답변을 위해 더 세분화된 data source를 읽는 것을 선호할 수 있습니다. 이는 예상된 동작입니다. Agent는 자연스럽게 권위 있는 data를 찾습니다. Agent가 상위 수준 CLAUDE.md context를 사용하게 하려면 명시적인 prompt 지침을 사용하세요(아래 예제 참고). 여기서 중요한 교훈을 얻습니다. CLAUDE.md는 data source에 대한 엄격한 제약이 아니라 *context와 지침*을 제공합니다.

In [2]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",  # CLAUDE.md가 있는 하위 디렉터리 지정
        setting_sources=["project"],
    )
) as agent:
    await agent.query("What's our current runway?")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# HTML rendering으로 응답 표시
display_agent_response(messages)
# 이 prompt로 agent는 CLAUDE.md 값인 burn 약 $500K, runway 20개월을 사용해야 함

🤖 Thinking...


#### Agent의 Data Source 선호 이해하기

**방금 일어난 일:**
Prompt에   를 추가하여, agent가 CSV 파일에서 더 세분화된 data를 찾는 대신 CLAUDE.md context를 따르도록 안내했습니다.

**핵심 Insight:**
1. **제약이 아닌 Context로서의 CLAUDE.md**: `cwd`를 설정하면 CLAUDE.md 파일이 배경 context로 load됩니다. 하지만 agent는 자연스럽게 사용 가능한 가장 권위 있는 data source를 찾습니다. 상세 CSV 파일이 있으면 정확성을 위해 이를 선호할 수 있습니다.

2. **Prompt Engineering이 중요합니다**: "high-level financial numbers from context"라는 표현은 financial_data/burn_rate.csv의 정확한 월별 data($525K gross, $235K net burn) 대신 CLAUDE.md의 단순한 경영진용 요약($500K burn, 20개월 runway)을 원한다는 신호를 agent에 줍니다.

3. **Architecture 설계 선택**: 이 동작은 production system에서 사실 바람직합니다. Agent가 최선의 data source를 찾기를 원하기 때문입니다. CLAUDE.md에는 다음이 포함되어야 합니다:
   - 상위 수준 context 및 전략
   - 회사 정보 및 standard
   - 상세 data 위치를 가리키는 정보
   - 상위 수준 수치와 상세 수치를 언제 사용할지에 대한 지침

4. **실제 Pattern**: CLAUDE.md를 agent에게 방향을 알려 주는 "onboarding 문서"로, 상세 파일을 정확성이 중요할 때 agent가 query할 수 있는 "source system"으로 생각하세요.

### 기능 1: Python Script 실행을 위한 Bash Tool

**무엇**: Bash tool은 agent가 여러 작업 중에서도 Python script를 직접 실행할 수 있게 하여, agent 본연의 기능을 넘어서는 절차적 지식, 복잡한 계산, data 분석 및 기타 통합을 제공합니다.

**왜**: Chief of Staff는 data 파일 처리, 재무 model 실행, data 기반 시각화 생성이 필요할 수 있습니다. 모두 Bash tool을 사용하기에 좋은 시나리오입니다.

**어떻게**: Agent가 접근할 수 있는 위치에 Python script를 두고, 무엇이며 어떻게 호출할 수 있는지에 대한 context를 추가하세요. Chief of Staff agent용 script라면 CLAUDE.md 파일에 context를 추가하고, subagent용이라면 해당 MD 파일에 추가하세요(뒤에서 자세히 설명). 이 tutorial에서는 `chief_of_staff_agent/scripts`에 다섯 개의 간단한 예제를 추가했습니다:
1. `hiring_impact.py`: 새로운 engineering 인력 채용이 burn rate, runway, 현금 보유액에 미치는 영향을 계산합니다. `financial-analyst` subagent가 월 $500K burn 및 20개월 runway를 기준으로 채용 시나리오를 model링하는 데 필수적입니다.
2. `talent_scorer.py`: 가중 기준으로 후보자의 기술 역량, 경험, 문화 적합성, 희망 연봉을 평가합니다. `recruiter` subagent가 TechStart의 senior engineer 기준 연봉 $180-220K에 맞춰 engineering 후보자를 순위화하는 핵심 tool입니다.
3. `simple_calculation.py`: Runway, burn rate, 분기 metrics를 빠르게 재무 계산합니다. Chief of Staff가 복잡한 model링 없이 즉시 metrics를 얻는 utility script입니다.
4. `financial_forecast.py`: 현재 $2.4M ARR이 MoM 15% 성장한다는 조건에서 ARR 성장 시나리오(기본/낙관/비관)를 model링합니다. `financial-analyst`가 Series B 준비 수준을 예측하고 $30M 자금 조달 목표를 검증하는 데 필수적입니다.
5. `decision_matrix.py`: SmartDev 인수나 사무실 확장 같은 전략적 선택의 가중 decision matrix를 생성합니다. Chief of Staff가 여러 이해관계자 및 기준이 있는 복잡한 결정을 체계적으로 평가하도록 돕습니다.

In [3]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        allowed_tools=["Bash", "Read"],
        cwd="chief_of_staff_agent",  # Agent가 정의된 하위 디렉터리 지정
    )
) as agent:
    await agent.query(
        "Use your simple calculation script with a total runway of 2904829 and a monthly burn of 121938."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# HTML rendering으로 응답 표시
display_agent_response(messages)

🤖 Using: Glob()
🤖 Using: Glob()
🤖 Using: Glob()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Read()
✓ Tool completed
🤖 Thinking...
🤖 Using: Bash()
✓ Tool completed
🤖 Thinking...


Metric,Value
Total Runway,"$2,904,829.00"
Monthly Burn,"$121,938.00"
Runway Months,~23.82 months
Quarterly Burn,"$365,814.00"
Daily Burn Rate,"$4,064.60"


### 기능 2: Output Style

**무엇**: Output style을 통해 대상에 따라 서로 다른 출력 style을 사용할 수 있습니다. 각 style은 Markdown 파일에 정의됩니다.

**왜**: Agent 사용자의 전문성 수준이나 우선순위가 다를 수 있습니다. 별도 agent를 만들지 않고도 output style로 이 집단들을 구별할 수 있습니다.

**어떻게**:
- `chief_of_staff_agent/.claude/output-styles/`에 style별 Markdown 파일을 구성합니다. 예를 들어 `.claude/output-styles/executive.md`의 Executive Output style을 확인하세요. Output style은 name 및 description 두 field를 포함하는 간단한 frontmatter로 정의됩니다. 참고: frontmatter의 name이 파일 이름과 정확히 일치하는지 확인하세요(대소문자 구분)

> **중요**: Output style은 Claude Code 내부 system prompt를 수정합니다. Software engineering에 초점을 맞춘 부분을 제외하여 software engineering을 넘어서는 특정 사용 사례를 더 잘 제어할 수 있게 합니다.

> **SDK 구성 참고**: Slash command(기능 4에서 다룸)와 마찬가지로 output style은 filesystem의 `.claude/output-styles/`에 저장됩니다. SDK가 이 파일을 load하려면 `ClaudeAgentOptions`에 **반드시** `setting_sources=["project"]`를 포함해야 합니다. `settings` parameter는 *어떤* style을 사용할지 알려 주지만, style 정의를 실제로 *load*하려면 `setting_sources`가 필요합니다. 이 요구사항은 이후 절을 debugging하면서 확인되었으며 모든 filesystem 기반 설정에 적용됩니다.

In [4]:
messages_executive = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        settings='{"outputStyle": "executive"}',
        # 중요: .claude/output-styles/의 output style을 load하려면 setting_sources에 "project"가 있어야 함
        # 이것이 없으면 SDK는 filesystem 설정(output style, slash command 등)을 load하지 않음
        setting_sources=["project"],
    )
) as agent:
    await agent.query("Tell me in two sentences about your writing output style.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages_executive.append(msg)

messages_technical = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        settings='{"outputStyle": "technical"}',
        setting_sources=["project"],
    )
) as agent:
    await agent.query("Tell me in two sentences about your writing output style.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages_technical.append(msg)

🤖 Thinking...
🤖 Thinking...


In [5]:
# Executive style 응답 표시
display_agent_response(messages_executive)

In [6]:
# Technical output style - 상세하고 구현 중심
display_agent_response(messages_technical, title="Technical Style")

### 기능 3: Plan Mode - 실행하지 않고 전략적으로 계획하기

**무엇**: Plan mode는 어떤 행동도 수행하지 않고 상세 실행 계획을 생성하도록 agent에 지시합니다. Agent는 요구사항을 분석하고 해결책 및 단계를 제시하지만 파일 수정, command 실행 또는 변경은 하지 않습니다.

**왜**: 복잡한 작업은 사전 계획을 통해 오류를 줄이고, 검토를 가능하게 하며, 조정을 개선할 수 있습니다. 계획 단계 이후 agent는 실행 내내 따라갈 일관된 흐름을 갖습니다.

**어떻게**: `permission_mode="plan"`만 설정하세요.

**계획의 영속 저장**: 계획은 검토 및 의사결정에 유용한 artifact이므로, 이를 가져와 영속적인 Markdown 파일로 저장하는 방법을 보여 줍니다. 이해관계자는 실행 승인 전에 계획을 검토할 수 있습니다.

> 참고: 이 기능은 Claude Code에서는 빛을 발하지만 SDK의 headless application에 완전히 맞추려면 아직 작업이 필요합니다. 특히 agent는 interactive mode에서만 의미가 있는 `ExitPlanMode()` tool을 호출하려 합니다. 이 경우 `continue_conversation=True`로 후속 query를 보내 agent가 context 안에서 계획을 실행하게 할 수 있습니다.

In [7]:
# =============================================================================
# Plan Mode helper 함수
# =============================================================================
# 이 utility는 agent가 계획을 출력할 수 있는 다양한 방식을 처리합니다.
# Agent는 직접 text, Write tool 또는 Claude의
# 내부 계획 디렉터리로 출력할 수 있으므로 여러 출처에서 견고하게 추출해야 합니다.

import glob as glob_module
import os
import re
from datetime import datetime
from pathlib import Path
from typing import Any


def extract_plan_from_xml(text: str | None, min_length: int = 200) -> str | None:
    """
    Text의 <plan> tag 사이에서 콘텐츠를 추출합니다.

    Args:
        text: 계획 콘텐츠를 검색할 text
        min_length: 유효한 계획의 최소 문자 수(빈 match 방지)

    Returns:
        추출한 계획 콘텐츠. 없거나 너무 짧으면 None
    """
    if not text:
        return None
    match = re.search(r"<plan>(.*?)</plan>", text, re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        if len(extracted) > min_length:
            return extracted
    return None


def extract_plan_from_messages(
    plan_content: list[str], min_fallback_length: int = 500
) -> tuple[str | None, str | None]:
    """
    수집한 message stream 콘텐츠에서 계획 추출을 시도합니다.

    Args:
        plan_content: Streaming 중 수집한 text block 목록
        min_fallback_length: Fallback의 최소 길이(XML tag 없음)

    Returns:
        (plan_text, source_description) tuple
    """
    combined_text = "\n\n".join(plan_content)

    # 첫 시도: XML tag
    plan = extract_plan_from_xml(combined_text)
    if plan:
        return plan, "message stream"

    # Fallback: 내용이 충분하면 raw 콘텐츠 사용
    if len(combined_text.strip()) > min_fallback_length:
        return combined_text.strip(), "full message content (fallback)"

    return None, None


def extract_plan_from_write_tool(
    write_contents: list[str], min_fallback_length: int = 500
) -> tuple[str | None, str | None]:
    """
    수집한 Write tool 호출에서 계획 추출을 시도합니다.

    Args:
        write_contents: Write tool 호출의 content 문자열 목록
        min_fallback_length: Fallback의 최소 길이(XML tag 없음)

    Returns:
        (plan_text, source_description) tuple
    """
    for content in write_contents:
        # 먼저 XML 추출 시도
        plan = extract_plan_from_xml(content)
        if plan:
            return plan, "Write tool capture"

        # Fallback: tag 없는 충분한 콘텐츠
        if content and len(content.strip()) > min_fallback_length:
            return content.strip(), "Write tool capture (no XML tags)"

    return None, None


def extract_plan_from_claude_dir(
    max_age_seconds: int = 300, min_fallback_length: int = 500
) -> tuple[str | None, str | None]:
    """
    Claude 내부 계획 디렉터리에서 최근 생성한 계획을 확인합니다.

    Args:
        max_age_seconds: 검토할 계획 파일의 최대 경과 시간(기본값: 5분)
        min_fallback_length: Fallback의 최소 길이(XML tag 없음)

    Returns:
        (plan_text, source_description) tuple
    """
    claude_plans_dir = os.path.expanduser("~/.claude/plans")

    if not os.path.exists(claude_plans_dir):
        return None, None

    # 가장 최근 계획 파일 찾기
    plan_files = sorted(
        glob_module.glob(os.path.join(claude_plans_dir, "*.md")),
        key=os.path.getmtime,
        reverse=True,
    )

    if not plan_files:
        return None, None

    most_recent = plan_files[0]
    file_age = datetime.now().timestamp() - os.path.getmtime(most_recent)

    if file_age > max_age_seconds:
        return None, None

    with open(most_recent) as f:
        content = f.read()

    filename = os.path.basename(most_recent)

    # 먼저 XML 추출 시도
    plan = extract_plan_from_xml(content)
    if plan:
        return plan, f"Claude plan file ({filename})"

    # Fallback: tag 없는 충분한 콘텐츠
    if len(content.strip()) > min_fallback_length:
        return content.strip(), f"Claude plan file ({filename}, no XML tags)"

    return None, None


def save_plan_to_file(
    plan_content: str,
    plan_source: str,
    model_name: str,
    prompt_summary: str,
    output_dir: str = "chief_of_staff_agent/plans",
    title: str = "Agent Plan: Engineering Restructure for AI Focus",
) -> Path:
    """
    추출한 계획을 timestamp가 있는 Markdown 파일에 저장합니다.

    Args:
        plan_content: 저장할 계획 text
        plan_source: 계획을 추출한 출처 설명
        model_name: 계획 생성에 사용한 model
        prompt_summary: 원래 prompt의 간단한 설명
        output_dir: 계획 파일을 저장할 디렉터리
        title: 계획 문서의 title

    Returns:
        저장한 계획 파일의 경로
    """
    plans_dir = Path(output_dir)
    plans_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    plan_file = plans_dir / f"plan_{timestamp}.md"

    with open(plan_file, "w") as f:
        f.write(f"# {title}\n\n")
        f.write(f"**Created:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"**Prompt:** {prompt_summary}\n")
        f.write(f"**Model:** {model_name}\n")
        f.write(f"**Plan Source:** {plan_source}\n\n")
        f.write("---\n\n")
        f.write(plan_content)
        f.write("\n\n---\n\n")
        f.write("*This plan was generated in plan mode and has not been executed.*\n")

    return plan_file


def capture_message_content(
    msg: Any,
    plan_content: list[str],
    write_tool_content: list[str],
    write_tool_paths: list[str],
) -> None:
    """
    Streaming message를 처리하고 관련 계획 콘텐츠를 수집합니다.

    이 function은 세 가지 가능한 출처에서 콘텐츠를 추출합니다:
    1. Message 콘텐츠의 text block
    2. Write tool 호출 parameter
    3. 최종 result attribute

    Args:
        msg: Agent stream의 message 객체
        plan_content: Text 콘텐츠를 append할 목록
        write_tool_content: Write tool 콘텐츠를 append할 목록
        write_tool_paths: Write tool 파일 경로를 append할 목록
    """
    # 출처 1: message 콘텐츠의 text block
    if hasattr(msg, "content"):
        for block in msg.content:
            if hasattr(block, "text"):
                plan_content.append(block.text)

            # 출처 2: Write tool 호출
            if hasattr(block, "type") and block.type == "tool_use":
                if hasattr(block, "name") and block.name == "Write":
                    if hasattr(block, "input") and isinstance(block.input, dict):
                        if "content" in block.input:
                            write_tool_content.append(block.input["content"])
                        if "file_path" in block.input:
                            write_tool_paths.append(block.input["file_path"])

    # 출처 3: 최종 결과
    if hasattr(msg, "result") and msg.result:
        plan_content.append(msg.result)


print("✅ Plan Mode helper functions loaded")

✅ Plan Mode helper functions loaded


In [8]:
# =============================================================================
# Plan Mode 구성
# =============================================================================

# 참고: MODEL은 cell-0에서 "claude-opus-4-6"으로 정의됨
# Opus는 복잡한 계획 작업에 탁월함

# Prompt는 다음을 위해 신중히 작성되었습니다:
# 1. 명시적인 context 제공(Opus가 명시적인 정보를 선호하므로)
# 2. 신뢰할 수 있는 추출을 위해 XML tag가 있는 출력 요청
# 3. Programmatic하게 계획을 수집하도록 파일 쓰기 방지

PLAN_PROMPT = """Restructure our engineering team for AI focus.

**CONTEXT (from CLAUDE.md):**
You are the Chief of Staff for TechStart Inc, a 50-person B2B SaaS startup that raised $10M Series A.
- Current engineering team: 25 people (Backend: 12, Frontend: 8, DevOps: 5)
- Monthly burn rate: ~$500K, Runway: 20 months
- Senior Engineer compensation: $180K-$220K + equity

**CRITICAL OUTPUT INSTRUCTIONS:**

1. **DO NOT use the Write tool** - Output your plan directly in your response text
2. **DO NOT save to any files** - I will handle saving the plan myself
3. **Wrap your ENTIRE plan inside `<plan> </plan>` XML tags** in your response

**Required Format:**
<plan>
[Your complete restructuring plan here - include all sections, timelines, budgets, and recommendations]
</plan>

**IMPORTANT:**
- The plan content MUST appear directly in your response between the XML tags
- Do NOT use Write, Edit, or any file-saving tools
- You may research and analyze before outputting, but the final plan must be in your response text
- Include: team structure, hiring recommendations, timeline, budget impact, and success metrics
- Use the company context provided above - do NOT ask clarifying questions"""

print(f"📋 Plan Mode configured with model: {MODEL}")
print(f"📝 Prompt length: {len(PLAN_PROMPT):,} characters")

📋 Plan Mode configured with model: claude-opus-4-6
📝 Prompt length: 1,180 characters


In [9]:
# =============================================================================
# Plan Mode Agent 실행
# =============================================================================
# Plan mode를 활성화하여 agent를 실행합니다. Agent는 상세 계획을 생성하지만
# 어떤 행동도 실행하지 않습니다. 서로 다른 agent 동작을 처리하도록
# 여러 출처에서 콘텐츠를 수집합니다.

# 수집 목록 초기화
messages = []
plan_content = []  # Message stream의 text
write_tool_content = []  # Write tool 호출의 content
write_tool_paths = []  # Write tool 호출의 경로

# Agent를 plan mode로 실행
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        permission_mode="plan",
        cwd="chief_of_staff_agent",
    )
) as agent:
    await agent.query(PLAN_PROMPT)
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

        # 이 message에서 콘텐츠 수집
        capture_message_content(msg, plan_content, write_tool_content, write_tool_paths)

print(f"\n✅ Agent completed. Captured {len(plan_content)} content blocks.")

🤖 Thinking...
🤖 Using: ExitPlanMode()
✓ Tool completed
🤖 Thinking...

✅ Agent completed. Captured 3 content blocks.


In [10]:
# =============================================================================
# 계획 추출 및 저장
# =============================================================================
# 계획 콘텐츠를 찾기 위해 우선순위 순서로 여러 출처를 시도합니다.
# 서로 다른 agent 동작을 견고하게 처리합니다.

final_plan = None
plan_source = None

# 우선순위 1: Message stream(선호 - agent 응답에서 직접 수집)
final_plan, plan_source = extract_plan_from_messages(plan_content)

# 우선순위 2: Write tool 수집(지침에도 불구하고 agent가 저장한 경우)
if not final_plan and write_tool_content:
    final_plan, plan_source = extract_plan_from_write_tool(write_tool_content)

# 우선순위 3: Claude 내부 계획 디렉터리(안전망)
if not final_plan:
    final_plan, plan_source = extract_plan_from_claude_dir()

# 결과 보고
if final_plan:
    print(f"✅ Plan extracted from: {plan_source}")
    print(f"   Plan length: {len(final_plan):,} characters")

    # 파일에 저장
    plan_file = save_plan_to_file(
        plan_content=final_plan,
        plan_source=plan_source,
        model_name=MODEL,
        prompt_summary="Restructure our engineering team for AI focus.",
    )
    print(f"\n📁 Plan saved to: {plan_file}")
else:
    error_msg = "Could not extract plan content from any source!\n"
    error_msg += "   Sources checked: message stream, Write tool, ~/.claude/plans/"
    if write_tool_paths:
        error_msg += f"\n   Write tool attempted to save to: {write_tool_paths}"
    print(f"❌ ERROR: {error_msg}")
    raise RuntimeError(f"Plan extraction failed: {error_msg}")

✅ Plan extracted from: message stream
   Plan length: 6,783 characters

📁 Plan saved to: chief_of_staff_agent/plans/plan_20251204_152737.md


In [11]:
# Style이 적용된 HTML로 계획 결과 표시
display_agent_response(messages, title="Engineering Restructure Plan")

#### 저장한 계획 실행하기

앞서 설명한 대로 agent는 계획을 만든 뒤 중지합니다. 저장한 계획 파일은 이해관계자용 검토 artifact 역할을 합니다.

**검토 후 계획을 실행하려면:**
1. `chief_of_staff_agent/plans/plan_*.md`에 저장한 계획을 검토합니다
2. 승인되면 `continue_conversation=True`로 새 query를 보내고 `permission_mode="plan"`을 제거하여 실행합니다

이 workflow는 "계획 → 검토 → 승인 → 실행" cycle을 가능하게 하며, 조직 개편이나 대규모 infrastructure 변경처럼 중요한 결정에 적합합니다.

#### 계획의 영속 저장 방식

위 코드에서는 Plan Mode agent가 계획을 출력하는 다양한 방식을 처리하는 **견고한 다중 출처 계획 수집 체계**를 구현했습니다:

**과제:**
`permission_mode="plan"`을 사용할 때 agent는 계획을 서로 다른 방식으로 출력할 수 있습니다:
1. Message stream에 **직접 text 출력**(이상적인 경우)
2. `~/.claude/plans/`에 저장하는 **Write tool**(Claude 내부 계획 system)
3. Custom 경로에 저장하는 **Write tool**

수집 체계는 **우선순위 기반 fallback system**으로 세 시나리오를 모두 처리합니다:

**출처 우선순위(순서대로):**

1. **Message Stream**(선호)
   - Streaming 중 `msg.content`에서 text block 수집
   - `<plan></plan>` XML tag 사이의 콘텐츠 추출
   - 콘텐츠가 응답에서 직접 오므로 가장 깔끔한 방식

2. **Write Tool 수집**
   - Message stream에서 Write tool 호출 모니터링
   - 작성하는 `content` parameter 추출
   - Prompt 지침에도 불구하고 agent가 저장하기로 결정할 때 유용

3. **Claude 내부 계획 디렉터리**
   - `~/.claude/plans/`에서 최근(5분 이내) 생성한 계획 파일 확인
   - 가장 최근 파일의 콘텐츠를 읽고 추출
   - 다른 방법이 실패할 때 안전망 역할

4. **전체 콘텐츠 Fallback**
   - XML tag가 없지만 상당한 콘텐츠(500자 초과)가 있으면 직접 사용
   - 부분 정보를 보존하면서 빈 계획 파일 방지

**핵심 구현 세부사항:**

```python
def extract_plan_from_text(text):
    """<plan> tag 사이의 콘텐츠를 추출하며, 없거나 비어 있으면 None을 반환합니다."""
    match = re.search(r'<plan>(.*?)</plan>', text, re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        # 최소 콘텐츠 길이 검증(실제 계획은 내용이 충분해야 함)
        if len(extracted) > 200:
            return extracted
    return None
```

**콘텐츠 검증이 중요한 이유:**
- 이전 version은 콘텐츠 없이 추출에 "성공"하면 빈 계획 파일을 만들 수 있었습니다
- 이제 XML tag로 둘러싼 콘텐츠에 최소 200자를 요구합니다
- Regex가 빈 콘텐츠나 사소한 콘텐츠에 match하는 false positive를 방지합니다

**직접 출력을 위한 Prompt Engineering:**
Prompt는 agent에 명시적으로 지시합니다:
- **Write tool을 사용하지 마세요** - filesystem을 통한 우회 방지
- **응답에 직접 출력하세요** - 콘텐츠가 message stream을 통해 흐르게 함
- **XML tag를 사용하세요** - 장황할 수 있는 응답에서 깔끔한 추출 가능

이 방식은 다음을 제공합니다:
- **신뢰성**: Agent 동작과 관계없이 계획 수집
- **투명성**: 저장된 파일에 사용한 출처 표시
- **Audit Trail**: Timestamp 및 출처 metadata가 있는 모든 계획의 기록
- **Debugging**: 추출 실패 시 명확한 오류 message

저장한 계획을 보겠습니다:

In [15]:
# Markdown rendering으로 저장한 계획 표시
from IPython.display import Markdown, display

# 올바른 Markdown formatting으로 계획 표시
display(Markdown(f"## 📋 Saved Plan Preview\n\n{final_plan}"))

print(f"\n📁 Full plan with metadata saved to: {plan_file}")

## 📋 Saved Plan Preview

# TechStart Inc. Engineering Team Restructuring Plan for AI Focus

## Executive Summary
Restructure the 25-person engineering team to build AI/ML capabilities while maintaining core product development. This plan balances immediate hiring needs with internal upskilling and strategic reorganization over a 12-month period.

---

## Current State Assessment

| Team | Headcount | Current Focus |
|------|-----------|---------------|
| Backend | 12 | Core product, APIs, infrastructure |
| Frontend | 8 | Web/mobile interfaces |
| DevOps | 5 | CI/CD, cloud infrastructure |
| **Total** | **25** | |

**Key Constraints:**
- Monthly burn: ~$500K | Runway: 20 months
- Senior Engineer comp: $180K-$220K + equity
- Series A stage - need to show growth metrics

---

## Proposed Future State (Month 12)

### New Team Structure

| Team | Current | Future | Change |
|------|---------|--------|--------|
| Backend/Core | 12 | 8 | -4 |
| Frontend | 8 | 6 | -2 |
| DevOps/MLOps | 5 | 6 | +1 |
| **AI/ML Platform** | 0 | 5 | +5 |
| **AI Product** | 0 | 4 | +4 |
| **Total** | **25** | **29** | **+4** |

### New Teams Created

**1. AI/ML Platform Team (5 engineers)**
- Focus: Infrastructure, model training pipelines, ML tooling
- Composition: 2 new hires (ML Engineers) + 2 internal transfers (Backend) + 1 internal (DevOps)
- Lead: New hire - Senior ML Engineer ($200-220K)

**2. AI Product Team (4 engineers)**
- Focus: AI features, integrations, user-facing ML applications
- Composition: 2 new hires (ML/AI specialists) + 1 internal (Backend) + 1 internal (Frontend)
- Lead: Promoted internal senior engineer + AI upskilling

---

## Hiring Plan

### New Positions (6 total new hires)

| Role | Priority | Timeline | Comp Range | Notes |
|------|----------|----------|------------|-------|
| Senior ML Engineer (Lead) | P0 | Month 1-2 | $200-220K | Team lead, architecture |
| ML Engineer | P0 | Month 2-3 | $180-200K | Platform focus |
| ML Engineer | P1 | Month 3-4 | $180-200K | Product focus |
| AI/ML Engineer | P1 | Month 4-5 | $170-190K | Generalist |
| MLOps Engineer | P2 | Month 5-6 | $160-180K | DevOps + ML |
| Junior ML Engineer | P2 | Month 6-8 | $130-150K | Growth hire |

**Total New Hiring Cost:** ~$1.02-1.14M annually

### Internal Transfers & Upskilling (4 engineers)

| From Team | # Engineers | New Role | Training Investment |
|-----------|-------------|----------|---------------------|
| Backend | 3 | ML Platform/Product | $15K each (courses, certs) |
| Frontend | 1 | AI Product (UI/UX) | $10K (AI tools training) |

**Upskilling Budget:** ~$55K total

---

## Implementation Timeline

### Phase 1: Foundation (Months 1-3)
- [ ] Hire Senior ML Engineer (Team Lead) - **Critical first hire**
- [ ] Identify 4 internal transfer candidates based on interest/aptitude
- [ ] Begin ML upskilling program for transfer candidates
- [ ] Set up ML infrastructure foundations (MLOps engineer involvement)
- [ ] Define AI product roadmap with Product team

### Phase 2: Team Formation (Months 4-6)
- [ ] Complete core AI team hiring (4 of 6 hires)
- [ ] Officially launch AI/ML Platform team
- [ ] Internal transfers complete bootcamp/training
- [ ] First AI feature POC delivered
- [ ] MLOps practices integrated into DevOps workflow

### Phase 3: Scaling (Months 7-9)
- [ ] Complete remaining hires
- [ ] AI Product team fully operational
- [ ] First AI feature in production
- [ ] Cross-team collaboration patterns established
- [ ] Model monitoring and observability in place

### Phase 4: Optimization (Months 10-12)
- [ ] Team velocity optimization
- [ ] Evaluate team composition effectiveness
- [ ] Plan for next growth phase
- [ ] Document AI development best practices

---

## Budget Impact Analysis

### Monthly Cost Changes

| Category | Current | Month 6 | Month 12 |
|----------|---------|---------|----------|
| Engineering Salaries | ~$375K | ~$430K | ~$485K |
| Training/Upskilling | $0 | $9K | $2K |
| AI Tools/Infrastructure | ~$5K | ~$20K | ~$35K |
| Recruiting Costs | Variable | ~$40K | ~$15K |
| **Monthly Total** | **~$380K** | **~$499K** | **~$537K** |

### Annual Impact Summary
- **Year 1 Additional Investment:** ~$900K-1.1M
- **New Monthly Burn (Month 12):** ~$537K (+$37K from engineering growth)
- **Runway Impact:** Reduces to ~17 months (still healthy for Series A)

### ROI Considerations
- AI features typically command 20-40% pricing premium in B2B SaaS
- Competitive differentiation in market
- Potential for AI-driven operational efficiencies

---

## Risk Mitigation

| Risk | Likelihood | Impact | Mitigation |
|------|------------|--------|------------|
| Senior ML hire takes >3 months | Medium | High | Engage specialized recruiters early; consider contractor bridge |
| Internal transfers struggle with ML | Low | Medium | Rigorous selection process; extended training runway |
| AI projects don't deliver value | Medium | High | Start with high-impact, lower-complexity features |
| Team culture friction | Low | Medium | Integrate teams gradually; shared goals and rituals |
| Budget overrun | Medium | Medium | Phase hiring based on runway checks quarterly |

---

## Success Metrics

### 6-Month Milestones
- [ ] AI/ML Platform team fully staffed (5 engineers)
- [ ] First AI-powered feature in beta
- [ ] 4 internal engineers completed ML certification
- [ ] ML infrastructure supporting model training/deployment

### 12-Month Milestones
- [ ] 2+ AI features in production
- [ ] AI team velocity matches core team benchmarks
- [ ] Customer NPS improvement attributable to AI features
- [ ] 15%+ of product roadmap is AI-focused
- [ ] Retention of transferred engineers >90%

### KPIs to Track
- Time-to-hire for ML roles
- AI feature adoption rates
- Model performance metrics (latency, accuracy)
- Engineering velocity (story points/sprint) across teams
- Employee satisfaction scores (especially transfers)

---

## Immediate Next Steps (Week 1-2)

1. **Executive alignment** - Present plan to leadership for approval
2. **Recruiter engagement** - Brief recruiting on Senior ML Engineer search
3. **Internal survey** - Gauge interest in AI/ML roles among current engineers
4. **Budget approval** - Finance sign-off on increased burn rate
5. **Infrastructure assessment** - DevOps audit of ML infrastructure needs

---

## Organizational Chart (Future State)

```
VP Engineering
├── Backend/Core Team (8)
│   └── 2 squads × 4 engineers
├── Frontend Team (6)
│   └── 2 squads × 3 engineers
├── DevOps/MLOps Team (6)
│   └── 4 DevOps + 2 MLOps
├── AI/ML Platform Team (5) [NEW]
│   └── Lead + 4 engineers
└── AI Product Team (4) [NEW]
    └── Lead + 3 engineers
```

---

*Plan prepared for TechStart Inc. Series A stage (~$10M raised, 20-month runway)*
*Recommended review: Quarterly budget and hiring progress checkpoints*


📁 Full plan with metadata saved to: chief_of_staff_agent/plans/plan_20251204_152737.md


## 고급 기능

### 기능 4: Custom Slash Command

> 참고: Slash command는 새로운 agent 기능이 아니라 사용자를 위한 syntactic sugar입니다

**무엇**: Custom slash command는 사용자 단축 구문(예: `/budget-impact`)으로 실행할 수 있는 사전 정의 prompt template입니다. Agent 기능이 아닌 **사용자용 shortcut**입니다. 완전하고 잘 작성한 prompt로 확장되는 keyboard shortcut이라고 생각하세요.

**왜**: Chief of Staff는 반복적인 경영진 질문을 처리합니다. 사용자는 복잡한 prompt를 반복 입력하는 대신 이미 검증된 prompt를 사용할 수 있습니다. 이를 통해 일관성과 표준화가 개선됩니다.

**어떻게**:
- `.claude/commands/`에 Markdown 파일을 정의합니다. 예를 들어 `.claude/commands/slash-command-test.md`에 하나를 정의했습니다. Command 정의 방식에 주목하세요. 두 field(name, description)의 frontmatter와, query로 전달한 argument를 포함할 수 있는 확장 prompt로 구성됩니다.
- `$ARGUMENTS`(전체 argument 문자열) 또는 `$1`, `$2` 등(positional argument)으로 prompt에 parameter를 추가할 수 있습니다
- 사용자는 prompt에서 slash command를 사용합니다

> **중요한 SDK 구성**: SDK를 사용할 때 slash command가 작동하려면 `ClaudeAgentOptions`에 **반드시** `setting_sources=["project"]`를 설정해야 합니다. 기본적으로 SDK는 isolation mode에서 작동하며 filesystem 설정(slash command, CLAUDE.md, subagent, hook 등)을 load하지 않습니다. 이는 이런 설정을 자동으로 load하는 Claude Code의 interactive 사용과 다릅니다.

In [16]:
# 사용자 입력: "/slash-command-test this is a test"
# -> 내부적으로 .claude/commands/slash-command-test.md의 prompt로 확장됨
# 여기서는 확장된 prompt가 문장의 단어 순서를 단순히 뒤집도록 지시함

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        # 중요: .claude/commands/의 slash command를 load하려면 setting_sources에 "project"가 있어야 함
        # 이것이 없으면 SDK는 filesystem 설정(slash command, CLAUDE.md 등)을 load하지 않음
        setting_sources=["project"],
    )
) as agent:
    await agent.query("/slash-command-test this is a test")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Thinking...


In [17]:
display_agent_response(messages, title="Slash Command Result")

### 기능 5: Hook - 자동화된 Deterministic Action

**무엇**: Hook은 다른 event뿐 아니라 특정 tool 호출 전(pre)이나 후(post)에 자동으로 실행하도록 설정할 수 있는 Python script입니다. Hook은 **deterministic하게** 실행되므로 검증 및 audit trail에 적합합니다.

**왜**: Agent에 guardrail을 두거나(예: 위험한 작업 방지) audit trail을 남기고 싶은 상황을 생각해 보세요. Hook은 agent가 작업을 달성할 자유를 충분히 허용하면서 안전하게 동작하는지 보장하는 데 agent와 함께 사용하기에 적합합니다.

**어떻게**:
- `.claude/hooks/`에 hook script 정의 -> hook이 발생했을 때 실행할 동작이 _무엇인지_
- `.claude/settings.local.json`에 hook 구성 정의 -> hook이 _언제_ 발생해야 하는지
- 여기서는 특정 tool 호출(Bash, Write, Edit)을 감시하도록 hook을 구성합니다
- 해당 tool을 호출하면 tool 완료 후 hook script가 실행됩니다(PostToolUse)

> **SDK 구성 참고**: `.claude/settings.local.json`의 hook에는 `setting_sources=["project", "local"]`이 필요합니다. SDK는 세 가지 setting source를 구별합니다:
> - `"project"` → `.claude/settings.json`(version control에 포함, 팀 공유)
> - `"local"` → `.claude/settings.local.json`(gitignore에 포함, hook 같은 로컬 설정)
> - `"user"` → `~/.claude/settings.json`(global 사용자 설정)
>
> Hook이 `settings.local.json`에 있으므로 `setting_sources`에 `"local"`을 포함해야 합니다.

**예제: Compliance를 위한 Report 추적**

Audit 및 compliance를 위해 재무 보고서의 Write/Edit 작업을 기록하는 hook입니다.
Hook은 `chief_of_staff_agent/.claude/hooks/report-tracker.py`에 정의하고 이를 적용하는 로직은 `chief_of_staff_agent/.claude/settings.local.json`에 있습니다:


```json
"hooks": {
  "PostToolUse": [
    {
      "matcher": "Write",
      "hooks": [
        {
          "type": "command",
          "command": "$CLAUDE_PROJECT_DIR/.claude/hooks/report-tracker.py"
        }
      ]
    },
    {
      "matcher": "Edit",
      "hooks": [
        {
          "type": "command",
          "command": "$CLAUDE_PROJECT_DIR/.claude/hooks/report-tracker.py"
        }
      ]
    }
  ]
}
```

In [18]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="chief_of_staff_agent",
        allowed_tools=["Bash", "Write", "Edit", "MultiEdit"],
        # 중요: Hook을 load하려면 setting_sources에 "project"와 "local" 둘 다 있어야 함
        # - "project"는 .claude/settings.json을 load함(공유 설정, CLAUDE.md, slash command)
        # - "local"은 .claude/settings.local.json을 load함(hook 구성 위치)
        setting_sources=["project", "local"],
    )
) as agent:
    await agent.query(
        "Create a quick Q2 financial forecast report with our current burn rate and runway projections. Save it to our /output_reports folder."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# Hook은 이를 audit/report_history.json에 추적 기록함
display_agent_response(messages, title="Q2 Financial Forecast")

🤖 Using: Bash()
🤖 Using: Bash()
🤖 Using: Bash()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Bash()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Write()
✓ Tool completed
🤖 Thinking...


Metric,Value
Cash in Bank,$10M
Current Runway,20 months (until Sept 2025)
Monthly Burn,$500K gross / ~$260K net
Q2 Total Net Burn,$740K
ARR,$2.4M (15% MoM growth)


이제 `./chief_of_staff_agent/audit/report_history.json`을 보면 agent가 보고서를 생성하거나 변경한 사실이 기록되어 있습니다. 생성된 보고서 자체는 `./chief_of_staff_agent/output_reports/`에서 볼 수 있습니다.

### 기능 6: Task Tool을 통한 Subagent

**무엇**: Task tool은 agent가 전문 작업을 다른 subagent에 위임할 수 있게 합니다. 각 subagent는 자체 지침, tool, 전문 지식을 갖습니다.

**왜**: Subagent를 추가하면 많은 가능성이 열립니다:
1. 전문화: 각 subagent는 자기 분야의 전문가입니다
2. 별도 context: Subagent는 자체 대화 기록과 tool을 갖습니다
3. 병렬화: 여러 subagent가 서로 다른 측면을 동시에 작업할 수 있습니다.

**어떻게**:
- allowed_tools에 `"Task"` 추가
- System prompt로 agent에 작업 위임 방법을 지시(CLAUDE.md에 더 일반적으로 정의할 수도 있음)
- `.claude/agents/`에 agent별 Markdown 파일 생성. 예를 들어 `.claude/agents/financial-analyst.md`를 보고, (sub)agent가 얼마나 쉽고 직관적인 Markdown 파일로 정의되는지 확인하세요. 세 field(name, description, tools)의 frontmatter와 system prompt로 구성됩니다. Description은 주 Chief of Staff agent가 각 subagent를 언제 호출할지 아는 데 유용합니다.

**시각화 개선**: Subagent 작업이 명확히 보이도록 `print_activity()` 및 `visualize_conversation()` utility를 개선했습니다:
- 🚀는 subagent에 위임할 때 표시됩니다(subagent 이름 포함)
- 📎는 subagent가 사용하는 tool을 나타냅니다(시각적 계층을 위해 들여쓰기)
- 시각적 구분선은 subagent 위임 및 완료 경계를 명확히 표시합니다
- 대화 timeline에 작업 설명과 prompt를 표시합니다

In [19]:
# 새로운 query를 시작하기 전에 subagent 추적 context 초기화
# Activity 표시에 깨끗한 state를 보장함
reset_activity_context()

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        allowed_tools=["Task"],  # Chief agent의 subagent 호출을 활성화함
        system_prompt="Delegate financial questions to the financial-analyst subagent. Do not try to answer these questions yourself.",
        cwd="chief_of_staff_agent",
        setting_sources=["project", "local"],
    )
) as agent:
    await agent.query("Should we hire 5 engineers? Analyze the financial impact.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

display_agent_response(messages, title="Hiring Impact Analysis")

🤖 Thinking...
🚀 Delegating to subagent: financial-analyst
   └─ Task: Analyze 5 engineer hiring impact
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   📎 [financial-analyst] Using: Read()
   📎 [financial-analyst] Using: Read()
   ✓ Tool completed
   ✓ Tool completed
   ✓ Tool completed
   📎 [financial-analyst] Using: Read()
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   📎 [financial-analyst] Using: Bash()
   ✓ Tool completed
   ✓ Tool completed
   📎 [financial-analyst] Thinking...


Metric,Current,After 5 Hires,Change
Monthly Burn,"$500,000","$575,833",+15.2%
Runway,20 months,17.1 months,-2.9 months
Headcount,50,55,+10%
Engineering %,50%,54.5%,+4.5%
Scenario,Mix,New Monthly Burn,New Runway
A (Recommended),3 Senior + 2 Junior,"$575,833",17.1 months
B,5 Senior,"$591,665",16.6 months
C,2 Senior + 3 Junior,"$567,917",17.3 months


In [20]:
visualize_conversation(messages)

Metric,Current,After 5 Hires,Change
Monthly Burn,"$500,000","$575,833",+15.2%
Runway,20 months,17.1 months,-2.9 months
Headcount,50,55,+10%
Engineering %,50%,54.5%,+4.5%
Scenario,Mix,New Monthly Burn,New Runway
A (Recommended),3 Senior + 2 Junior,"$575,833",17.1 months
B,5 Senior,"$591,665",16.6 months
C,2 Senior + 3 Junior,"$567,917",17.3 months


여기서 주 agent가 subagent를 사용하기로 결정하면:
  1. 다음과 같은 parameter로 Task tool을 호출합니다:
  ```json
    {
      "description": "Analyze hiring impact",
      "prompt": "Analyze the financial impact of hiring 5 engineers...",
      "subagent_type": "financial-analyst"
    }
  ```
  2. Task tool이 별도 context에서 subagent를 실행합니다
  3. 계속 처리하도록 결과를 주 Chief of Staff agent에 반환합니다

## 모두 결합하기

이제 살펴본 모든 것을 결합하겠습니다. Agent에 senior engineer 3명을 채용하는 재무 영향을 판단하고 insight를 `output_reports/hiring_decision.md`에 작성하도록 요청합니다. 이는 위의 모든 기능을 보여 줍니다:
- **Bash Tool**: 새로운 engineer 채용의 영향을 판단하기 위해 `hiring_impact.py` script를 실행하는 데 사용
- **Memory**: 현재 예산, runway, 매출 및 기타 관련 정보를 이해하기 위해 디렉터리의 `CLAUDE.md`를 context로 읽음
- **Output style**: `chief_of_staff_agent/.claude/output-styles`에 정의된 다양한 output style
- **Custom Slash Command**: `chief_of_staff_agent/.claude/commands`에 정의된 전체 prompt로 확장되는 `/budget-impact` shortcut 사용
- **Subagent**: `/budget_impact` command가 Chief of Staff agent에 `chief_of_staff_agent/.claude/agents`의 financial-analyst subagent를 호출하도록 안내
- **Hook**: `chief_of_staff_agent/.claude/hooks`에 정의하고 `chief_of_staff_agent/.claude/settings.local.json`에 구성
    - Agent가 재무 보고서를 갱신하면 hook은 `chief_of_staff_agent/audit/report_history.json` logfile에 edit/write activity를 기록해야 함
    - Financial analyst subagent가 `hiring_impact.py` script를 호출하면 `chief_of_staff_agent/audit/tool_usage_log.json` logfile에 기록됨

- **Plan Mode**: Chief of Staff가 행동 전에 승인할 계획을 작성하게 하려면 아래 주석 처리된 줄을 해제하세요

즉시 사용할 수 있도록 이전 Notebook과 마찬가지로 agent loop를 Python 파일에 encapsulate했습니다. `chief_of_staff_agent` 하위 디렉터리의 agent.py를 확인하세요.

전체적으로 `send_query()` function은 4개 parameter(prompt, continue_conversation, permission_mode, output_style)를 받으며, 나머지(system prompt, max turn, allowed tool, 작업 디렉터리)는 모두 agent 파일에 설정됩니다.

이들이 어떻게 결합되는지 더 잘 시각화하려면 [Claude가 만들어 준 flow 및 architecture diagram :)](./chief_of_staff_agent/flow_diagram.md)을 확인하세요.


In [21]:
from chief_of_staff_agent.agent import send_query

reset_activity_context()

result, messages = await send_query(
    "/budget-impact hiring 3 senior engineers. Save your insights by updating the 'hiring_decision.md' file in /output_reports or creating a new file there",
    # permission_mode="plan", # Planning mode를 사용하려면 활성화
    output_style="executive",
)

🤖 Thinking...
🤖 Using: Glob()
🤖 Using: Glob()
✓ Tool completed
✓ Tool completed
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Task()
✓ Tool completed
🤖 Using: Bash()
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Write()
✓ Tool completed
🤖 Thinking...


In [22]:
visualize_conversation(messages)

Metric,Current,Post-Hiring,Change
Monthly Gross Burn,$525K,$590K,+$65K (+12.4%)
Monthly Net Burn,$235K,$300K,+$65K (+27.7%)
Cash Runway,42.6 months,32.9 months,-9.7 months
Break-Even,-,November 2024,5 months


## 결론

Claude Code SDK로 enterprise 수준 기능을 갖춘 정교한 multi-agent system을 구축하는 방법을 보여 주었습니다. Bash tool의 기본 script 실행부터 시작하여 CLAUDE.md의 영속적인 memory, 대상별 custom output style, 전략적인 planning mode, 사용자 편의를 위한 slash command, guardrail용 compliance hook, 전문 작업을 위한 subagent 조정을 점진적으로 소개했습니다.

이 기능들을 결합하여 복잡한 경영진 의사결정 workflow를 처리할 수 있는 AI Chief of Staff를 만들었습니다. System은 재무 분석을 전문 subagent에 위임하고, hook으로 audit trail을 유지하며, 이해관계자에 따라 소통 style을 조정하고, data 기반 분석에 뒷받침된 실행 가능한 insight를 제공합니다.

고급 agentic pattern 및 multi-agent orchestration의 기반은 production용 enterprise system 구축을 준비하는 데 도움이 됩니다. 다음 Notebook에서는 Model Context Protocol(MCP) server를 통해 agent를 외부 service에 연결하여 내장 tool을 넘어 기능을 크게 확장하는 방법을 살펴봅니다.

다음: [02_The_observability_agent.ipynb](02_The_observability_agent.ipynb) - MCP를 통해 custom 통합 및 외부 data source로 agent를 확장하는 방법을 배우세요.